# 03 — FINAL Held-out Evaluation and RQ1–RQ4 (REVIEW)

Run only after a clean full run of **Notebook 02 FINAL**. It imports the frozen candidate configurations and feature sets. Final test quarters are used **once**, never to choose models or thresholds. The primary population is all eligible company quarters; the training-derived minimum-current-PAT rule defines a separately reported sensitivity cohort.

Outputs: frozen holdout predictions, MAE/RMSE/R², company-block paired intervals, model-specific permutation importance, algorithm comparison, reported-vs-engineered and seasonality comparisons, a secondary PAT-eligibility experiment, research question findings, and provenance. No numerical conclusions are prefilled.


## Load frozen manifests and verify split integrity

In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

SEED=640
SOURCE=Path('statistical_outputs')
if not (SOURCE/'prepared_company_quarters.csv').exists(): SOURCE=Path('/content/statistical_outputs')
assert (SOURCE/'prepared_company_quarters.csv').exists(), 'Run Notebook 01 and place statistical_outputs beside this notebook.'
prepared=SOURCE/'prepared_company_quarters.csv'
handoff=json.loads((SOURCE/'handoff_manifest.json').read_text())
source_hash=hashlib.sha256(prepared.read_bytes()).hexdigest()
assert source_hash==handoff['prepared_csv_sha256'], 'Input file checksum mismatch'
df=pd.read_csv(prepared,low_memory=False)
# Generate *as-of-quarter* changes using the EXACT preceding calendar quarter.
# This function uses source financial values and identifiers only; NEVER PAT_next/target_growth_pct.
QOQ_CORE = ['PAT','REVENUE','TOTAL_INCOME','OTHER_INCOME','PROFIT_BEFORE_TAX',
            'TAX_EXPENSE','EXPENSES','FINANCE_COSTS','EPS_BASIC','EPS_DILUTED',
            'ASSETS','LIABILITIES','EQUITY','BORROWINGS']
QOQ_BANK = ['OPERATING_PROFIT_BEFORE_PROVISIONS','INTEREST_EARNED',
            'INTEREST_EXPENDED','PROVISIONS','GROSS_NPA','NET_NPA']
QOQ_SOURCE_COLUMNS = QOQ_CORE + QOQ_BANK
QOQ_SUFFIXES = ['_qoq_abs_change', '_qoq_growth_pct']

def add_exact_quarter_growth(panel):
    required = {'COMPANY_KEY','q'}
    assert required.issubset(panel.columns), 'Prepared dataset must have COMPANY_KEY and q.'
    out = panel.copy()
    out['_q_period'] = pd.PeriodIndex(out['q'].astype(str),freq='Q')
    assert not out.duplicated(['COMPANY_KEY','_q_period']).any(), \
        'Ambiguous company-quarter rows: fix source before computing QoQ.'
    present = [x for x in QOQ_SOURCE_COLUMNS if x in out.columns]
    for x in present:
        out[x] = pd.to_numeric(out[x], errors='coerce').replace([np.inf,-np.inf],np.nan)
    # Drop any pre-existing generated fields, to guarantee deterministic recomputation.
    for x in present:
        out.drop(columns=[x+s for s in QOQ_SUFFIXES],inplace=True,errors='ignore')
    # Shift the previous quarter's key FORWARD by one quarter, then join to current.
    # This prevents false QoQ growth across gaps such as Q1 -> Q3.
    prev = out[['COMPANY_KEY','_q_period']+present].copy()
    prev['_q_period'] = prev['_q_period']+1
    prev.rename(columns={x:'_previous_'+x for x in present},inplace=True)
    out=out.merge(prev,on=['COMPANY_KEY','_q_period'],how='left',validate='one_to_one',sort=False)
    for x in present:
        previous=out.pop('_previous_'+x)
        current=out[x]
        out[x+'_qoq_abs_change']=current-previous
        # For losses and negative base values, use abs(previous), matching the study's target convention.
        denominator=previous.abs().where(previous.abs()>0)
        out[x+'_qoq_growth_pct']=100*(current-previous)/denominator
    out.drop(columns=['_q_period'],inplace=True)
    new_fields=[x+s for x in present for s in QOQ_SUFFIXES]
    out[new_fields]=out[new_fields].replace([np.inf,-np.inf],np.nan)
    return out, new_fields

# Same-calendar-quarter-last-year features; never use future quarter values.
SEASONAL_SOURCE_COLUMNS = QOQ_SOURCE_COLUMNS
SEASONAL_SUFFIXES = ['_lag4', '_yoy_abs_change', '_yoy_growth_pct']
SEASONAL_CALENDAR = ['target_quarter_sin', 'target_quarter_cos']

def add_exact_annual_seasonality(panel):
    out = panel.copy()
    out['_season_q'] = pd.PeriodIndex(out['q'].astype(str),freq='Q')
    assert not out.duplicated(['COMPANY_KEY','_season_q']).any(), 'Duplicate company-quarter values'
    present = [c for c in SEASONAL_SOURCE_COLUMNS if c in out.columns]
    for c in present:
        out[c] = pd.to_numeric(out[c], errors='coerce').replace([np.inf,-np.inf], np.nan)
    drop_cols = [c+s for c in present for s in SEASONAL_SUFFIXES] + SEASONAL_CALENDAR
    out.drop(columns=drop_cols, errors='ignore', inplace=True)
    lag = out[['COMPANY_KEY','_season_q']+present].copy()
    lag['_season_q'] = lag['_season_q'] + 4
    lag.rename(columns={c:'_lag4_source_'+c for c in present}, inplace=True)
    out = out.merge(lag, on=['COMPANY_KEY','_season_q'],how='left', validate='one_to_one',sort=False)
    for c in present:
        prev = out.pop('_lag4_source_'+c)
        cur = out[c]
        out[c+'_lag4'] = prev
        out[c+'_yoy_abs_change'] = cur-prev
        out[c+'_yoy_growth_pct'] = 100*(cur-prev)/prev.abs().where(prev.abs()>0)
    # The date whose PAT growth is being predicted is q+1, never read its value.
    tq = (out['_season_q']+1).dt.quarter.astype(float)
    out['target_quarter_sin'] = np.sin(2*np.pi*tq/4)
    out['target_quarter_cos'] = np.cos(2*np.pi*tq/4)
    out.drop(columns=['_season_q'],inplace=True)
    generated = [c+s for c in present for s in SEASONAL_SUFFIXES] + SEASONAL_CALENDAR
    out[generated] = out[generated].replace([np.inf,-np.inf],np.nan)
    return out, generated

df, QOQ_GENERATED = add_exact_quarter_growth(df)
df, SEASONAL_GENERATED = add_exact_annual_seasonality(df)
assert {'split','COMPANY_KEY','target_q','target_growth_pct','INDEX_INDUSTRY','quarter_of_year'}<=set(df)
df=df[df.split.isin(['train','validation','test'])].copy()
assert df.target_growth_pct.notna().all()
q={s:set(df.loc[df.split.eq(s),'target_q'].astype(str)) for s in ('train','validation','test')}
assert all(q.values()) and max(q['train'])<min(q['validation'])<min(q['test'])
assert not any(q[a]&q[b] for a,b in [('train','validation'),('train','test'),('validation','test')])
MODEL_OUT=SOURCE.parent/'model_outputs'
assert (MODEL_OUT/'selection_manifest.json').exists(),'Run Notebook 02 first.'
selection=json.loads((MODEL_OUT/'selection_manifest.json').read_text())
assert selection['source_sha256']==source_hash
OUT=SOURCE.parent/'final_evaluation_outputs'; OUT.mkdir(exist_ok=True,parents=True)
print('Prepared rows:', len(df), 'Reserved test-quarter labels:', selection['test_target_quarters'])

## Refit frozen models and compute held-out predictions

In [ ]:
from sklearn.linear_model import HuberRegressor
CATS=['INDEX_INDUSTRY','quarter_of_year']
config=selection.get('qoq_feature_config')
if config:
 assert config['source_columns']==QOQ_SOURCE_COLUMNS and config['suffixes']==QOQ_SUFFIXES, \
     'QoQ generator does not match frozen selection manifest.'
season_cfg=selection.get('seasonality_feature_config')
if season_cfg:
 assert season_cfg['source_columns']==SEASONAL_SOURCE_COLUMNS
 assert season_cfg['suffixes']==SEASONAL_SUFFIXES
 assert season_cfg['calendar_columns']==SEASONAL_CALENDAR
assert set(sum([list(x) for x in selection['feature_sets'].values()],[])).issubset(df.columns), \
    'Model features missing after QoQ regeneration.'
for c in sorted(set(sum([list(x) for x in selection['feature_sets'].values()],[]))):
 if c in CATS:df[c]=df[c].fillna('Unknown').astype(str)
 else:df[c]=pd.to_numeric(df[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
assert set(df.loc[df.split.eq('test'),'target_q'].astype(str))==set(selection['test_target_quarters'])
assert selection['source_sha256']==source_hash
assert set(df.loc[df.split.eq('train'),'target_q'].astype(str))==set(selection['train_target_quarters'])
assert set(df.loc[df.split.eq('validation'),'target_q'].astype(str))==set(selection['validation_target_quarters'])
dev=df[df.split.isin(['train','validation'])].copy()
test=df[df.split.eq('test')].copy()
def pipe_for(name,params,features):
 nums=[x for x in features if x not in CATS]
 cats=[x for x in features if x in CATS]
 try: encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
 except TypeError:encoder=OneHotEncoder(handle_unknown='ignore',sparse=False)
 preprocessing=ColumnTransformer([('numeric',Pipeline([('impute',SimpleImputer(strategy='median')),
  ('scale',StandardScaler())]),nums),('categorical',encoder,cats)])
 if name=='Historical median':model=DummyRegressor(**params)
 elif name=='Linear Regression':model=LinearRegression()
 elif name=='Ridge':model=Ridge(**params)
 elif name=='Random Forest':model=RandomForestRegressor(random_state=SEED,n_jobs=-1,**params)
 elif name=='Gradient Boosting':model=GradientBoostingRegressor(random_state=SEED,**params)
 elif name=='GB absolute error':model=GradientBoostingRegressor(loss='absolute_error',random_state=SEED,**params)
 elif name=='Huber':model=HuberRegressor(**params)
 else:raise ValueError(name)
 return Pipeline([('preprocess',preprocessing),('model',model)])
def parsed_key(key):
 fs,algo,p=key.split('|',2)
 return fs,algo,json.loads(p)
def metrics(y,p):return {'MAE':float(mean_absolute_error(y,p)),
  'RMSE':float(np.sqrt(mean_squared_error(y,p))),'R2':float(r2_score(y,p))}
keys=sorted(set([selection['primary']['key']]+list(selection['rq3_keys'].values())+
                list(selection['rq4_keys'].values())+list(selection.get('seasonality_keys',{}).values())))
models={};predictions={};results=[]
for key in keys:
 fs,algo,params=parsed_key(key)
 cols=selection['feature_sets'][fs]
 model=pipe_for(algo,params,cols);model.fit(dev[cols],dev.target_growth_pct)
 pred=model.predict(test[cols]);assert np.isfinite(pred).all()
 models[key]=model;predictions[key]=pred
 results.append({'key':key,'algorithm':algo,'feature_set':fs,**metrics(test.target_growth_pct,pred)})
score_table=pd.DataFrame(results).sort_values('MAE')
score_table.to_csv(OUT/'heldout_model_metrics.csv',index=False)
export=test[['COMPANY_KEY','target_q','target_growth_pct']].reset_index(drop=True).copy()
for k,key in enumerate(keys):export[f'prediction_{k:02d}']=predictions[key]
export.to_csv(OUT/'heldout_predictions.csv',index=False)
(OUT/'prediction_key_map.json').write_text(json.dumps({f'prediction_{i:02d}':key for i,key in enumerate(keys)},indent=2))
display(score_table)

## RQ1 — Frozen primary vs historical median (primary full population)

In [ ]:
def company_bootstrap(pred_a,pred_b,seed=640,reps=2000):
 y=test.target_growth_pct.to_numpy(float)
 delta=np.abs(y-pred_a)-np.abs(y-pred_b)
 grouped=pd.DataFrame({'firm':test.COMPANY_KEY.to_numpy(),'delta':delta})
 blocks=[g.delta.to_numpy() for _,g in grouped.groupby('firm',sort=False)]
 rng=np.random.default_rng(seed)
 draws=[np.concatenate([blocks[j] for j in rng.integers(0,len(blocks),len(blocks))]).mean()
       for _ in range(reps)]
 lo,hi=np.quantile(draws,[.025,.975])
 return {'MAE_improvement_pp':float(delta.mean()),'CI_low':float(lo), 'CI_high':float(hi),
         'companies':len(blocks),'quarters':int(test.target_q.nunique())}
primary=selection['primary']['key']
baseline=selection['rq3_keys']['Historical median']
rq1=company_bootstrap(predictions[baseline],predictions[primary]);rq1['primary_key']=primary
pd.DataFrame([rq1]).to_csv(OUT/'RQ1_baseline_comparison.csv',index=False)
display(pd.DataFrame([rq1]))
print('Company bootstrap is exploratory: few common time periods remain unaccounted for.')

## RQ2 — Primary model importance; if constant, a nonconstant frozen secondary comparator

In [ ]:
from sklearn.inspection import permutation_importance
chosen_key=primary
importance_role='primary'
if parsed_key(chosen_key)[1]=='Historical median':
    # A constant baseline has no feature attribution; use a frozen nonconstant comparator,
    # without suggesting that this changes the validation-selected primary winner.
    chosen_key=selection['rq3_keys'].get('GB absolute error')
    importance_role='secondary (primary is a constant baseline)'
assert chosen_key is not None
cols=selection['feature_sets'][parsed_key(chosen_key)[0]]
perm=permutation_importance(models[chosen_key],test[cols],test.target_growth_pct,
    scoring='neg_mean_absolute_error',n_repeats=15,random_state=SEED)
importance=pd.DataFrame({'feature':cols,'MAE_increase_pp':perm.importances_mean,
    'repeat_std':perm.importances_std,'model_key':chosen_key,'model_role':importance_role})
importance=importance.sort_values('MAE_increase_pp',ascending=False)
importance.to_csv(OUT/'RQ2_permutation_importance.csv',index=False)
display(importance)
print('Permutation importance is predictive reliance, not statistical association or causal importance.')

## RQ3 — Compare frozen candidate algorithms

In [ ]:
rows=[]
for algo,key in selection['rq3_keys'].items():
 comp=company_bootstrap(predictions[baseline],predictions[key])
 rows.append({'algorithm':algo,'key':key,**metrics(test.target_growth_pct,predictions[key]),**comp})
rq3_table=pd.DataFrame(rows).sort_values('MAE')
rq3_table.to_csv(OUT/'RQ3_model_comparison.csv',index=False)
display(rq3_table)
print('Test ranking is descriptive; the preselected primary model is NOT changed.')

## RQ4 — Controlled feature-set comparison within frozen robust Gradient Boosting family

In [ ]:
key_r=selection['rq4_keys']['reported_only']
key_e=selection['rq4_keys']['reported_plus_engineered']
assert parsed_key(key_r)[1]==parsed_key(key_e)[1]==selection['predeclared_rq4_algorithm']
rq4_res=company_bootstrap(predictions[key_r],predictions[key_e])
rq4_res.update({'model_family':selection['predeclared_rq4_algorithm'],'reported_key':key_r,'enhanced_key':key_e})
pd.DataFrame([rq4_res]).to_csv(OUT/'RQ4_feature_set_comparison.csv',index=False)
display(pd.DataFrame([rq4_res]))

## Optional banking-sector holdout comparison

In [ ]:
# Optional bank-only comparison from Notebook 02's frozen development selections.
info=selection.get('banking_comparison',{})
if info.get('status')=='selected':
 subset=df.INDEX_INDUSTRY.astype(str).str.strip().str.casefold().eq('financial services')
 bd=df[subset & df.split.isin(['train','validation'])].copy()
 bt=df[subset & df.split.eq('test')].copy()
 if len(bt)>0:
  bank_outputs={}
  for group,key in info['keys'].items():
   _,alg,params=parsed_key(key);cols=info['feature_sets'][group]
   for c in cols:
    if c in CATS:
     bd[c]=bd[c].fillna('Unknown').astype(str);bt[c]=bt[c].fillna('Unknown').astype(str)
    else:
     bd[c]=pd.to_numeric(bd[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
     bt[c]=pd.to_numeric(bt[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
   m=pipe_for(alg,params,cols);m.fit(bd[cols],bd.target_growth_pct)
   bank_outputs[group]=m.predict(bt[cols])
  bank_scores=pd.DataFrame([{'group':g,**metrics(bt.target_growth_pct,p)} for g,p in bank_outputs.items()])
  bank_scores.to_csv(OUT/'banking_heldout_metrics.csv',index=False);display(bank_scores)
 else: print('Financial Services holdout has no rows; banking comparison unavailable.')
else:print('Sector experiment not selected:',info.get('reason','Unavailable'))

## Supplementary seasonality experiment: same frozen robust GB family

In [ ]:
season_keys=selection.get('seasonality_keys',{})
if len(season_keys)==2:
    no_season=season_keys['reported_plus_engineered']
    with_season=season_keys['reported_plus_engineered_plus_seasonal']
    assert parsed_key(no_season)[1]==parsed_key(with_season)[1]==selection['predeclared_rq4_algorithm']
    seasonal_effect=company_bootstrap(predictions[no_season],predictions[with_season])
    seasonal_report=pd.DataFrame([{
        'without_seasonality_key':no_season,'with_seasonality_key':with_season,
        'without_seasonality_MAE':metrics(test.target_growth_pct,predictions[no_season])['MAE'],
        'with_seasonality_MAE':metrics(test.target_growth_pct,predictions[with_season])['MAE'],
        **seasonal_effect}])
    seasonal_report.to_csv(OUT/'seasonality_holdout_comparison.csv',index=False)
    display(seasonal_report)
else:
    print('No frozen seasonal comparison exists in manifest; run the seasonality-enabled Notebook 02.')

## Secondary sensitivity: minimum current-PAT eligibility (fixed from TRAIN only)

The threshold was frozen in Notebook 02 using **training-quarter current PAT**, not future growth. This supplementary evaluation (1) scores the existing full-population primary model on only eligible test rows; (2) retrains the **same algorithm and feature set** on eligible train+validation rows, then scores it on those identical eligible test rows. Report cohort size, percentage coverage and both MAEs. This is not the primary research population. The test target is used solely for evaluation and cannot alter eligibility, algorithm, threshold, or hyperparameters.

In [ ]:
threshold=float(selection['minimum_current_pat_threshold'])
assert threshold>=0 and np.isfinite(threshold)
eligible_dev=dev.loc[pd.to_numeric(dev.PAT,errors='coerce').abs().gt(threshold)].copy()
eligible_test=test.loc[pd.to_numeric(test.PAT,errors='coerce').abs().gt(threshold)].copy()
assert len(eligible_dev)>50 and len(eligible_test)>10, 'Too few PAT-eligible records; report unavailable.'
chosen_features=selection['feature_sets'][selection['primary']['feature_set']]
selected_fs,selected_name,selected_params=parsed_key(primary)
assert selected_fs==selection['primary']['feature_set']
restricted=pipe_for(selected_name,selected_params,chosen_features)
restricted.fit(eligible_dev[chosen_features],eligible_dev.target_growth_pct)
restricted_pred=restricted.predict(eligible_test[chosen_features])
# Predictions in primary array align with original test row order.
mask=pd.to_numeric(test.PAT,errors='coerce').abs().gt(threshold).to_numpy()
full_pred_on_eligible=predictions[primary][mask]
matched_eligible=pd.DataFrame([
 {'training_population':'full','evaluation_population':'eligible','train_n':len(dev),
  'test_n':len(eligible_test),'test_coverage':len(eligible_test)/len(test),
  **metrics(eligible_test.target_growth_pct,full_pred_on_eligible)},
 {'training_population':'eligible','evaluation_population':'eligible','train_n':len(eligible_dev),
  'test_n':len(eligible_test),'test_coverage':len(eligible_test)/len(test),
  **metrics(eligible_test.target_growth_pct,restricted_pred)}
])
matched_eligible.to_csv(OUT/'minimum_current_PAT_secondary_comparison.csv',index=False)
display(matched_eligible)
print('Threshold (source monetary units):',threshold)
print('This result is SECONDARY. The main RQ conclusions use the full-population test.')

## Final research evidence and caveats

In [ ]:
overview=pd.DataFrame([
 {'RQ':'RQ1','evidence':f"Historical-median vs frozen primary: MAE delta {rq1['MAE_improvement_pp']:.3f} pp",
  'qualification':'Short test horizon, exploratory company bootstrap'},
 {'RQ':'RQ2','evidence':'Model-specific test-set permutation analysis saved',
  'qualification':'Not causal; no importance claim for constant baseline'},
 {'RQ':'RQ3','evidence':f"Lowest descriptive test MAE: {rq3_table.iloc[0]['algorithm']}",
  'qualification':'Do not reselect primary based on test ranking'},
 {'RQ':'RQ4','evidence':f"Robust GB reported vs engineered MAE delta {rq4_res['MAE_improvement_pp']:.3f} pp",
  'qualification':'Controlled Ridge contrast; company bootstrap exploratory'}])
overview.to_csv(OUT/'RQ_final_evidence_summary.csv',index=False)
display(overview)
(OUT/'evaluation_provenance.json').write_text(json.dumps({'source_sha256':source_hash,
   'primary':primary,'test_quarters':sorted(test.target_q.astype(str).unique()),
   'test_rows':len(test),'selection_source':str(MODEL_OUT/'selection_manifest.json')},indent=2))

## Interpretation cautions

A very small number of test quarters limits temporal replication. Source XBRL tags, filing dates and exceptional values must be checked. Model importance does not prove causal influence; paired company bootstrap does not capture shared quarter shocks. Report unfavorable results as observed.